# 00 · Discover adapters and identify the configured NRC400

The adapter-list cell opens no ports. Use its metadata to configure the verified adapter in `config.toml` before running the connection cell. Match the returned model, serial and firmware to the equipment; discovery is not unit identification.

Read [the tutorial guide](README.md) and [physical commissioning](../docs/commissioning.md) first. Use one serial owner. These notebook outputs are intentionally empty; no bench result is implied.


## Environment

From the repository root, run:

```powershell
uv sync --locked
uv run --locked --with jupyterlab --with ipykernel python -m jupyterlab
```

Choose that environment's Python kernel and run top to bottom. Restart the kernel after driver changes. The next cell opens no serial port.


In [ ]:
import sys
from pathlib import Path

# Prefer this checkout when launched from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, REPO_ROOT.parent):
    if (candidate / "src/tark_nrc/controller.py").is_file():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
print(f"Python: {sys.executable}")
print(f"Repository/config base: {REPO_ROOT}")

## Imports


In [ ]:
from pathlib import Path

from serial.tools import list_ports

from tark_nrc import create_chiller

## List adapters (this section opens no serial port)


In [ ]:
print("Available serial adapters; no devices probed:")
ports = sorted(list_ports.comports(), key=lambda item: item.device)
for port in ports:
    print(f"  {port.device}: {port.description}")
    print(f"    USB serial: {port.serial_number or 'unknown'}; hardware ID: {port.hwid}")
if not ports:
    print("  No serial ports reported by the operating system.")

## Choose connection settings

Set `CONFIG_PATH` to your verified NRC400 profile. This keeps USB adapter/unit identity bindings and verified serial framing/timeouts together. The current `[control]` section accepts only `expected_serial_number`; remove old `allow_writes`, `min_c` and `max_c` entries. The API has no write-enable switch, target range or state-based control lock. Choose targets for the actual coolant and experiment. There is no guessed COM port. `SIMULATE = True` ignores the hardware profile and opens no physical port; each run creates a fresh simulated device with no shared state. Review all operation settings before continuing.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config.toml"
SIMULATE = False  # True uses an independent offline device and ignores CONFIG_PATH.

## Connect, identify and release

Run this entire operation cell together: connection and cleanup share one context/finally block. Ctrl+C/Interrupt attempts cleanup while allowing Python to finish. Do not split the cell, kill the kernel during I/O, or rerun setup over an unfinished connection. Errors remain visible. Closing the port retains the equipment's target, loop and pump state.


In [ ]:
# The production factory retains the profile's USB/unit binding and framing.
# Construction opens nothing; the context connects using identity queries only.
chiller = create_chiller(None if SIMULATE else CONFIG_PATH, simulate=SIMULATE)
with chiller:
    print(f"Backend: {chiller.transport.backend}; port: {chiller.transport.port}")
    print(f"Model: {chiller.identity.model}")
    print(f"Serial: {chiller.identity.serial_number}")
    print(f"Firmware: {chiller.identity.firmware}")

print("Connection closed. Equipment retains its target, loop and pump state.")

## Check results and finish

Record identity and firmware, then compare the sample with the panel in example 01.

**Software connection closure does not stop the chiller or pump.** Follow the physical shutdown procedure separately and confirm panel state. A forced kernel termination cannot guarantee cleanup. See [the hardware test record checklist](README.md#6-record-a-hardware-test-result) for evidence to retain.
